In [ ]:

import cv2
from onnx2torch import convert
import torch
from insightface.app import FaceAnalysis
import numpy as np

app = FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'], allowed_modules=["detection", "genderage"])
app.prepare(ctx_id=0)

onnx_model_path = app.models["detection"].model_file
torch_model = convert(onnx_model_path)

img = cv2.imread("data/images/000000531961.jpg")
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_resized = cv2.resize(img_rgb, (640, 640))

# InsightFace default preprocessing: just convert to float32
img_input = img_resized.astype(np.float32)
img_input = np.transpose(img_input, (2, 0, 1))  # (HWC) → (CHW)
img_input = np.expand_dims(img_input, axis=0)   # Add batch dim
img_input = torch.from_numpy(img_input)

bboxes, kpss = app.models["detection"].detect(img,
                                             metric='default')

In [ ]:

import cv2
from onnx2torch import convert
import torch
from insightface.app import FaceAnalysis
import numpy as np

app = FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'], allowed_modules=["detection", "genderage", "landmark_2d_106"])
app.prepare(ctx_id=0)

onnx_model_path = app.models["landmark_2d_106"].model_file
torch_model = convert(onnx_model_path)

print(torch_model)

# img = cv2.imread("data/images/000000531961.jpg")
# img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
# img_resized = cv2.resize(img_rgb, (640, 640))

# # InsightFace default preprocessing: just convert to float32
# img_input = img_resized.astype(np.float32)
# img_input = np.transpose(img_input, (2, 0, 1))  # (HWC) → (CHW)
# img_input = np.expand_dims(img_input, axis=0)   # Add batch dim
# img_input = torch.from_numpy(img_input)

# bboxes, kpss = app.models["detection"].detect(img,
#                                              metric='default')

In [ ]:
app.models["detection"].model_file


In [ ]:
from abc import ABC, abstractmethod
from typing import List, Tuple
import numpy as np

class FaceModelBase(ABC):
    """
    Abstract base class for face-related models.
    Enforces common interface across detection and attribute models.
    """

    @abstractmethod
    def prepare(self, *args, **kwargs):
        """Optional setup before prediction (e.g., thresholds, sizes)."""
        pass

    @abstractmethod
    def predict(self, img: np.ndarray, bboxes: List[Tuple[float, float, float, float]]) -> List:
        """
        Predict outputs based on image and (optional) bounding boxes.
        - For detection models: `bboxes` is ignored, return new bboxes.
        - For attribute models: `bboxes` are used to crop/align faces.
        """
        pass

In [ ]:
import cv2
import os
import numpy as np
import onnx
import torch
from onnx2torch import convert
from typing import Tuple, List
from insightface.utils.face_align import transform

class RetinaFaceTorch(FaceModelBase):
    """
    PyTorch-based port of the ONNXRuntime+NumPy RetinaFace wrapper.
    Usage:
        rft = RetinaFaceTorch(model_file="retinaface.onnx")
        rft.prepare(nms_thresh=0.4, det_thresh=0.5, input_size=(640,640))
        dets, kps = rft.detect(image)  # image as HxWxBGR uint8 numpy array
    """
    def __init__(self, model_file=None, input_size: Tuple[int, int] = (640, 640)):
        assert model_file is not None, "Must provide ONNX model file path"
        assert os.path.exists(model_file), f"Model file not found: {model_file}"
        # Convert ONNX to PyTorch
        onnx_model = onnx.load(model_file)
        self.torch_model = convert(onnx_model)
        self.torch_model.eval()
        # Set defaults
        self.model_file = model_file
        self.center_cache = {}
        self.nms_thresh = 0.4
        self.det_thresh = 0.5
        self.input_size = input_size
        self.input_mean = 127.5
        self.input_std = 128.0
        self.use_kps = False
        # Derive FPN settings by inspecting ONNX output count
        ort_sess = __import__('onnxruntime').InferenceSession(model_file, None)
        out_count = len(ort_sess.get_outputs())
        if out_count in (6, 9):
            self.fmc = 3
            self._feat_stride_fpn = [8, 16, 32]
        else:
            self.fmc = 5
            self._feat_stride_fpn = [8, 16, 32, 64, 128]
        # key 8/16/32/... anchors per stride
        # assume square anchor ratio 1.0, single anchor
        self._num_anchors = 1
        if out_count in (6, 9):
            self._num_anchors = 2
        if out_count in (9, 15):
            self.use_kps = True

    def prepare(self, nms_thresh=None, det_thresh=None, input_size=None):
        """Set thresholds and input canvas size (W,H)."""
        if nms_thresh is not None:
            self.nms_thresh = nms_thresh
        if det_thresh is not None:
            self.det_thresh = det_thresh
        if input_size is not None:
            self.input_size = input_size

    def forward(self, img):
        """
        Runs the PyTorch model on the preprocessed blob.
        Returns list of torch.Tensor outputs in the same order as ONNX.
        """
        H, W = img.shape[:2]
        # BlobFromImage: scale, mean/std normalization, swapRB
        blob = cv2.dnn.blobFromImage(
            img, 1.0/self.input_std, (W, H),
            (self.input_mean, self.input_mean, self.input_mean),
            swapRB=True
        )
        inp = torch.from_numpy(blob).float()
        with torch.no_grad():
            outs = self.torch_model(inp)
        # outs may be single tensor or list/tuple
        if isinstance(outs, torch.Tensor):
            outs = (outs,)
        return outs

    def detect(self, img, max_num=0, metric='default'):
        """
        Full detect pipeline: preprocess, forward, postprocess to bboxes/kps.
        img: HxWxBGR uint8
        returns: dets (N×5 numpy array: x1,y1,x2,y2,score), kps (N×5×2) or None
        """
        assert img.dtype == np.uint8, "Input must be uint8 BGR image"

        # Resize into input_size canvas
        if self.input_size is None:
            H, W = img.shape[:2]
            in_w, in_h = W, H
        else:
            in_w, in_h = self.input_size
        im_ratio = img.shape[0] / img.shape[1]
        model_ratio = in_h / in_w
        if im_ratio > model_ratio:
            new_h = in_h
            new_w = int(new_h / im_ratio)
        else:
            new_w = in_w
            new_h = int(new_w * im_ratio)

        
        resized = cv2.resize(img, (new_w, new_h))
        canvas = np.zeros((in_h, in_w, 3), dtype=np.uint8)
        canvas[:new_h, :new_w] = resized

        # Forward
        outs = self.forward(canvas)
        # Prepare lists
        scores_list, bboxes_list, kps_list = [], [], []

        # Split outputs
        # confidences: 0..fmc-1, bboxes: fmc..2*fmc-1, kps: 2*fmc.. if use_kps
        for idx, stride in enumerate(self._feat_stride_fpn):
            # Score
            sc = outs[idx].cpu().numpy().squeeze()
            # BBox preds
            bd = outs[idx + self.fmc].cpu().numpy()
            bd = bd * stride

            # KPS preds
            if self.use_kps:
                kp = outs[idx + self.fmc*2].cpu().numpy() * stride
                
            # Grid size
            height = canvas.shape[0] // stride
            width  = canvas.shape[1] // stride
            key = (height, width, stride)
           # Anchor centers *exactly* as in the ONNX wrapper*
            if key in self.center_cache:
                centers = self.center_cache[key]
            else:
                # 1) build a (height×width×2) array where [:,:,0]=x indices, [:,:,1]=y indices
                ac = np.stack(
                    np.mgrid[:height, :width][::-1],  # yields [x_grid, y_grid]
                    axis=-1
                ).astype(np.float32)
                # 2) scale by stride and flatten to (N,2)
                ac = (ac * stride).reshape(-1, 2)
                # 3) if using >1 anchor per location, replicate each center exactly
                if self._num_anchors > 1:
                    ac = np.stack([ac] * self._num_anchors, axis=1).reshape(-1, 2)
                centers = ac
                self.center_cache[key] = centers



            # Decode bboxes
            # distance2bbox: [x_center,y_center] + [l,t,r,b] -> [x1,y1,x2,y2]
            from insightface.model_zoo.retinaface import distance2bbox, distance2kps
            bbs = distance2bbox(centers, bd.reshape(-1, 4))
            # Filter by threshold
            pos = np.where(sc.ravel() >= self.det_thresh)[0]
            scores_list.append(sc.ravel()[pos])
            bboxes_list.append(bbs[pos])
            if self.use_kps:
                kpss = distance2kps(centers, kp.reshape(-1, kp.shape[-1]))
                kpss = kpss.reshape(-1, kp.shape[-1]//2, 2)
                kps_list.append(kpss[pos])

        # Stack results
        scores = np.vstack([s.reshape(-1,1) for s in scores_list])
        bboxes = np.vstack(bboxes_list)
        if self.use_kps:
            kpss = np.vstack(kps_list)
        else:
            kpss = None

        # Sort by score
        order = scores[:,0].argsort()[::-1]
        pre = np.hstack((bboxes, scores)).astype(np.float32)
        pre = pre[order]
        if kpss is not None:
            kpss = kpss[order]

        # Apply NMS
        keep = self.nms(pre)
        det = pre[keep]
        if kpss is not None:
            kpss = kpss[keep]

        # Clip and scale back to original image size
        scale = new_h / img.shape[0] if im_ratio>model_ratio else new_w / img.shape[1]
        det[:,:4] /= scale
        if kpss is not None:
            kpss /= scale

        # Optionally limit number of detections
        if max_num > 0 and len(det) > max_num:
            areas = (det[:,2]-det[:,0])*(det[:,3]-det[:,1])
            centers_img = np.array([(det[:,0]+det[:,2])/2, (det[:,1]+det[:,3])/2]).T
            img_center = np.array([img.shape[1]/2, img.shape[0]/2])
            dists = np.sum((centers_img - img_center)**2, axis=-1)
            metrics = areas - 2*dists if metric=='default' else areas
            idxs = np.argsort(metrics)[::-1][:max_num]
            det = det[idxs]
            if kpss is not None:
                kpss = kpss[idxs]

        return det, kpss

    
    def nms(self, dets):
        """Pure NumPy NMS."""
        x1, y1, x2, y2, scores = dets[:,0], dets[:,1], dets[:,2], dets[:,3], dets[:,4]
        areas = (x2 - x1 + 1) * (y2 - y1 + 1)
        order = scores.argsort()[::-1]
        keep = []
        while order.size > 0:
            i = order[0]
            keep.append(i)
            xx1 = np.maximum(x1[i], x1[order[1:]])
            yy1 = np.maximum(y1[i], y1[order[1:]])
            xx2 = np.minimum(x2[i], x2[order[1:]])
            yy2 = np.minimum(y2[i], y2[order[1:]])
            w = np.maximum(0.0, xx2 - xx1 + 1)
            h = np.maximum(0.0, yy2 - yy1 + 1)
            inter = w * h
            ovr = inter / (areas[i] + areas[order[1:]] - inter)
            inds = np.where(ovr <= self.nms_thresh)[0]
            order = order[inds + 1]
        return keep
    

    def predict(self, img: np.ndarray, bboxes=None) -> List[Tuple[float, float, float, float]]:
        """Wrapper around detect(), ignoring `bboxes` arg."""
        dets, kpss = self.detect(img)
        return dets.tolist(), kpss.tolist()


class AgeGenderModel(FaceModelBase):
    def __init__(self, model_path: str, det_size: Tuple[int, int] = (640, 640), face_size: Tuple[int, int] = (96, 96)):
        assert model_path is not None and os.path.exists(model_path), f"Model file not found: {model_path}"
        self.model_path = model_path
        # self.input_mean = 127.5
        # self.input_std = 128.0
        
        # important !!!
        self.input_mean = 0
        self.input_std = 1


        self.det_size = det_size
        self.face_size = face_size

        model = onnx.load(model_path)

        # Convert ONNX to Torch
        self.torch_model = convert(model)
        self.torch_model.eval()

    def prepare(self, *args, **kwargs):
        pass  # Nothing to prepare, or set `input_mean/std` dynamically here


    def preprocess_face(self, img: np.ndarray) -> torch.Tensor:
        img = cv2.resize(img, self.face_size)  # size = (W, H)

        # Convert BGR to RGB
        img = img[:, :, ::-1]  # OpenCV loads BGR; we want RGB

        # Convert to float32 and scale
        img = img.astype(np.float32)
        img = (img - self.input_mean) / self.input_std

        # HWC → CHW
        img = img.transpose(2, 0, 1)

        # Convert to tensor
        return torch.from_numpy(img)

    def predict(self, img: np.ndarray, bboxes: List[Tuple[float, float, float, float]]) -> List[Tuple[int, int]]:
        """
        Predict gender and age for each face in the given image.
        Args:
            img     : Original image (HWC, BGR or RGB)
            bboxes  : List of bounding boxes, each (x1, y1, x2, y2)
        Returns:
            List of (gender, age) tuples
        """
        results = []
        rotate = 0
        for bbox in bboxes:
            w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
            center = ((bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2)
            scale = self.face_size[0] / (max(w, h) * 1.5)


            aligned_face, _ = transform(img, center, self.face_size[0], scale, rotate)

            if aligned_face is None or aligned_face.size == 0:
                results.append((None, None))
                continue


            tensor = self.preprocess_face(aligned_face).unsqueeze(0)
            with torch.no_grad():
                output = self.torch_model(tensor)[0].cpu().numpy()

            gender = int(np.argmax(output[:2]))
            age = int(np.round(output[2] * 100))
            results.append((gender, age))
            

        return results

    

# Example usage:
image = cv2.imread("data/images/000000531961.jpg")

actual = app.get(image)
print(f"Actual predictions:")
print(f"Gender: {actual[0].gender}, Age: {actual[0].age}")
print(f"Bbox: {actual[0].bbox}")

rft = RetinaFaceTorch(onnx_model_path)
rft.prepare(nms_thresh=0.4, det_thresh=0.5, input_size=(640,640))
dets, kpss = rft.predict(image)
print("Detections:", dets)


age_model = AgeGenderModel(app.models["genderage"].model_file)
age_model.prepare()
preds = age_model.predict(image, dets)
print(preds)






In [ ]:
import torch
import hiddenlayer as hl

age_model = AgeGenderModel(model_path=app.models["genderage"].model_file)

# … your model definition …
dummy_input = torch.randn(1, 3, 96, 96)

# Build a HiddenLayer graph
hl_graph = hl.build_graph(age_model.torch_model, dummy_input)
hl_graph.theme = hl.graph.THEMES['blue']  # optional; you can pick any theme

# This writes out “hiddenlayer_graph.pdf” (or .png/.svg, etc.)
hl_graph.save("hiddenlayer_graph", format="pdf")


In [ ]:
import cv2
import numpy as np
from torchvision import transforms
import torch

img = cv2.imread("data/images/000000531961.jpg")
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_resized = cv2.resize(img_rgb, (640, 640))

# InsightFace default preprocessing: just convert to float32
img_input = img_resized.astype(np.float32)
img_input = np.transpose(img_input, (2, 0, 1))  # (HWC) → (CHW)
img_input = np.expand_dims(img_input, axis=0)   # Add batch dim
img_input = torch.from_numpy(img_input)


faces_insight = app.get(img)  # Uses full pipeline


bbox_insight = [face.bbox for face in faces_insight]

torch_model.eval()
with torch.no_grad():
    out = torch_model(img_input)

# For RetinaFace, this could be like: (loc, conf, iou, landmarks)
# You need to decode the output manually or use InsightFace decoding utils


In [ ]:
# import os
# import cv2
# import torch
# import onnx
# from onnx2torch import convert
# import numpy as np
# import pandas as pd
# from utils.config import load_config
# from tqdm import tqdm
# import torch.nn as nn

# # ─── 0) CONFIG & MODEL SETUP ─────────────────────────────────────────────────

# config = load_config("config.yaml")
# subj = 1

# # Load the list of image IDs
# faces_df        = pd.read_excel(f"data/labels/subj_{subj:02d}/faces/faces_final.xlsx")
# cocoIds         = faces_df["cocoId"].tolist()
# images_dir      = config.directories.images_target_dir
# activations_dir = config.directories.activations_dir

# # Initialize InsightFace analyzer
# allowed_modules = ['detection', 'genderage']  # später z.B. 'landmarks' ergänzen
# app = FaceAnalysis(
#     name='buffalo_l',
#     providers=['CPUExecutionProvider'],
#     allowed_modules=allowed_modules
# )
# app.prepare(ctx_id=0)

# # Convert ONNX → PyTorch once for each module and store
# torch_models = {}
# models_hooks = {}


# for module_name in allowed_modules:
#     onnx_path   = app.models[module_name].model_file
#     onnx_model  = onnx.load(onnx_path)
#     torch_model = convert(onnx_model)
#     torch_model.eval()
#     torch_models[module_name] = torch_model

# # ─── 1) PREPROCESS FUNCTION ──────────────────────────────────────────────────

# PIXEL_MEAN = (127.5, 127.5, 127.5)
# PIXEL_STD  = (128.0, 128.0, 128.0)
# INPUT_SIZE = (640, 640)

# def preprocess(img: np.ndarray) -> torch.Tensor:
#     blob = cv2.dnn.blobFromImage(
#         img,
#         scalefactor=1.0/PIXEL_STD[0],
#         size=INPUT_SIZE,
#         mean=PIXEL_MEAN,
#         swapRB=True
#     )
#     return torch.from_numpy(blob).float()

# # ─── 2) HOOK & EXTRACT ACTIVATIONS ────────────────────────────────────────────


# gt = app.get(img)
# print(f"Ground truth predictions:")
# print(f"Gender: {gt[0].gender}, Age: {gt[0].age}")

# for module_name, torch_model in torch_models.items():
#     print(f"\n=== Extracting from module: {module_name} ===")
    
#     # choose layers to hook (hier: alle ReLU und Sigmoid)
#     layers_to_hook = [
#         name for name, mod in torch_model.named_modules()
#         if isinstance(mod, (nn.ReLU, nn.Sigmoid))
#     ]

#     models_hooks[module_name] = layers_to_hook
    
#     activations = {}
#     hooks = []
#     def get_activation(name):
#         def hook(module, input, output):
#             activations[name] = output.detach().cpu()
#         return hook

#     for name, mod in torch_model.named_modules():
#         if name in layers_to_hook:
#             hooks.append(mod.register_forward_hook(get_activation(name)))

#     # test pass
#     img = cv2.imread(os.path.join(images_dir,       f"{cocoIds[0]:012d}.jpg"))
#     inp = preprocess(img)
#     with torch.no_grad():
#         pre = torch_model(inp)
#     # test pass
#     print(f"Test pass:\n{pre}")

#     # loop over images
#     for cocoId in tqdm(cocoIds, desc=f"{module_name} images"):
#         img_path  = os.path.join(images_dir,       f"{cocoId:012d}.jpg")
#         save_path = os.path.join(activations_dir,  module_name, f"subj_{subj:02d}", f"{cocoId:012d}.npz")

#         if os.path.exists(save_path):
#             continue
        
#         os.makedirs(os.path.dirname(save_path), exist_ok=True)

#         img = cv2.imread(img_path)
#         inp = preprocess(img)
#         with torch.no_grad():
#             _ = torch_model(inp)

#         # 3) Pull out NumPy arrays and clear for next loop
#         np_acts = {name: tensor.numpy() for name, tensor in activations.items()}

#         np.savez(save_path, **np_acts)

#         activations.clear()

#     # remove hooks
#     for h in hooks:
#         h.remove()
    
#     print(f"Finished module: {module_name}")







In [ ]:
from collections import defaultdict
import os
import cv2
import torch
import onnx
from onnx2torch import convert
import numpy as np
import pandas as pd
from utils.config import load_config
from tqdm import tqdm
import torch.nn as nn


# ─── CONFIG ──────────────────────────────────────────────────────────────

config         = load_config("config.yaml")
subj           = 1
images_dir     = config.directories.images_target_dir
activations_dir= config.directories.activations_dir
retina_path    = app.models["detection"].model_file
genderage_path = app.models["genderage"].model_file

faces_df = pd.read_excel(
    f"data/labels/subj_{subj:02d}/faces/faces_final.xlsx"
)
cocoIds  = faces_df["cocoId"].tolist()

# ─── ACTIVATION HOOK ──────────────────────────────────────────────────────

class ActivationHook:
    def __init__(self):
        self.features = defaultdict(list)

    def register_hooks(self, model: torch.nn.Module):
        handles = []
        for name, module in model.named_modules():
            if isinstance(module, (nn.ReLU, nn.Sigmoid)):
                handles.append(
                    module.register_forward_hook(
                        lambda mod, inp, out, name=name:
                            self.features[name].append(out.detach().cpu())
                    )
                )
        return handles

    def get_activations(self):
        single = {}
        for name, taps in self.features.items():
            if not taps: 
                continue
            # take the FIRST (or LAST) activation from the list
            tensor = taps[0]  
            single[name] = tensor.cpu().numpy()
        return single

    def clear(self):
        self.features.clear()



# ─── SETUP & HOOKS ────────────────────────────────────────────────────────

rft = RetinaFaceTorch(retina_path)
rft.prepare(nms_thresh=0.4, det_thresh=0.5)
age = AgeGenderModel(genderage_path)

retina_hook = ActivationHook()
age_hook    = ActivationHook()


retina_layers_to_hook = [
    name for name, mod in rft.torch_model.named_modules()
    if isinstance(mod, (nn.ReLU, nn.Sigmoid))
]

age_layers_to_hook = [
    name for name, mod in age.torch_model.named_modules()
    if isinstance(mod, (nn.ReLU, nn.Sigmoid))
]
# adjust these to real layer names in each .torch_model
# retina_layers_to_hook = ["body.stage1.0.conv1"]
# age_layers_to_hook    = ["conv_13_t0_relu"]

retina_handles = retina_hook.register_hooks(rft.torch_model)
age_handles    = age_hook.register_hooks(age.torch_model)

# ─── MAIN LOOP ────────────────────────────────────────────────────────────

for cocoId in tqdm(cocoIds):
    img_path = os.path.join(images_dir, f"{cocoId:012d}.jpg")
    img = cv2.imread(img_path)
    if img is None:
        print("Missing:", img_path)
        continue

    # 3) save both
    out_dir_detection = os.path.join(activations_dir, "detection", f"subj_{subj:02d}")
    out_dir_genderage = os.path.join(activations_dir, "genderage", f"subj_{subj:02d}")

    os.makedirs(out_dir_detection, exist_ok=True)
    os.makedirs(out_dir_genderage, exist_ok=True)

    if not os.path.exists(os.path.join(out_dir_detection,f"{cocoId:012d}.npz")):   
        # 1) detect & save retinaface activations
        dets, _ = rft.detect(img)
        _ = rft.forward(img)  # triggers hooks
        det_acts = retina_hook.get_activations()

        np.savez(os.path.join(out_dir_detection,f"{cocoId:012d}.npz"), **det_acts)
        retina_hook.clear()

    if not os.path.exists(os.path.join(out_dir_genderage,f"{cocoId:012d}.npz")):
        # 2) predict & save gender/age activations
        preds = age.predict(img, dets.tolist())
        # run through model to trigger hooks
        _ = age.predict(img, dets.tolist())
        age_acts = age_hook.get_activations()

        np.savez(os.path.join(out_dir_genderage,f"{cocoId:012d}.npz"), **age_acts)
        age_hook.clear()

# ─── CLEANUP ──────────────────────────────────────────────────────────────

for h in retina_handles + age_handles:
    h.remove()

In [ ]:
dict(rft.torch_model.named_parameters())

In [ ]:
import torch
from torchviz import make_dot

# Example dummy input
dummy_input = torch.randn(1, 3, 640, 640)  # RetinaFace input shape

# Forward pass
rft.torch_model.eval()
output = rft.torch_model(dummy_input)

# Create the dot graph (include model parameters for clearer view)
dot = make_dot(output[0], params=dict(rft.torch_model.named_parameters()))

# Save to a PDF
dot.render("retinaface_graph", format="pdf")

print("Graph saved as retinaface_graph.pdf")


In [ ]:
app.models["genderage"]

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from sklearn.manifold import MDS
import matplotlib.pyplot as plt

import seaborn as sns
from matplotlib.patches import Circle

# ——— set modern grid theme ———
sns.set_theme(style="whitegrid")

# Set manually
subj = 1
activations_dir = f"data/activations"  # adjust if needed
faces_df = pd.read_excel(f"data/labels/subj_{subj:02d}/faces/faces_final.xlsx")
cocoIds = faces_df["cocoId"].tolist()

allowed_modules = ['detection', 'genderage']
selected_layers = {
    "detection": ["Sigmoid_217"],
    "genderage": []
}
rsa_results = {}

for module in allowed_modules:
    print(f"Processing {module}")
    for layer in selected_layers[module]:
        print(f"Processing {layer}")
        features = []
        valid_ids = []

        # — load features for this layer —
        for cocoId in cocoIds:
            path = os.path.join(activations_dir, module,
                                f"subj_{subj:02d}", f"{cocoId:012d}.npz")
            if not os.path.exists(path):
                continue
            data = np.load(path)
            if layer not in data:
                continue
            features.append(data[layer].squeeze().flatten())
            valid_ids.append(cocoId)

        if len(features) < 2:
            print(f"Skipping {layer} due to insufficient data")
            continue

        # — compute MDS —
        X = np.vstack(features)
        rdm = squareform(pdist(X, metric="correlation"))
        mds = MDS(n_components=2, dissimilarity="precomputed", random_state=42)
        X_mds = mds.fit_transform(rdm)

        rsa_results[layer] = {"rdm": rdm, "mds": X_mds, "ids": valid_ids}

        # ——— plot MDS with unit circle & fixed limits ———
        fig, ax = plt.subplots(figsize=(6, 6))

        # draw unit circle
        circ = Circle((0, 0), radius=1.0,
                      fill=False, lw=1,
                      color='gray', alpha=0.7)
        ax.add_patch(circ)

        # scatter points
        ax.scatter(X_mds[:, 0], X_mds[:, 1],
                   c='k', s=30, edgecolors='w')

        # fix limits and aspect
        ax.set_xlim(-1, 1)
        ax.set_ylim(-1, 1)
        ax.set_aspect('equal', 'box')

        plt.title(f"MDS + bbox‐center for sigmoid 217")
        plt.xlabel("MDS 1")
        plt.ylabel("MDS 2")

        plt.tight_layout()
        plt.show()






In [ ]:
metadata = [f"{x:012d}" for x in cocoIds]


In [ ]:
X_mds.shape

In [ ]:
from rsa.permutation_analysis import run_mantel_test

In [ ]:
run_mantel_test(X_mds, 1, metadata, B=2000)

In [ ]:
run_mantel_test(X_mds, 1, metadata, B=2000, feature_selection="sizes")

In [ ]:
run_mantel_test(X_mds, 1, metadata, B=2000, feature_selection="feature_array_age")

In [ ]:
run_mantel_test(X_mds, 1, metadata, B=2000, feature_selection="genders")

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from sklearn.manifold import MDS
import matplotlib.pyplot as plt

# Set manually
subj = 1
activations_dir = f"data/activations/detection"
faces_df = pd.read_excel(f"data/labels/subj_{subj:02d}/faces/faces_final.xlsx")
cocoIds = faces_df["cocoId"].astype(int).tolist()

# load face detections and build a dict cocoId → center
face_det_path = f"data/labels/subj_{subj:02d}/face_detection_result.json"
with open(face_det_path, "r") as f:
    face_detections = json.load(f)

center_dict = {}
for det in face_detections:
    cid = det["file_name"][:-4]

    cid = int(cid.lstrip("0"))
    # print(f'{cid.lstrip("0")} {len(det["detection"])}')
    # only keep ones in our list and with exactly one bbox
    if cid in cocoIds and len(det["detection"]) == 1:
        x1, y1, x2, y2 = det["detection"][0]
        center_dict[cid] = np.array([ (x1 + x2) / 2, (y1 + y2) / 2 ])

print(center_dict)

selected_layers = ["Sigmoid_217"]
for layer in selected_layers:
    print(f"Processing {layer}")
    features, valid_ids = [], []
    for cocoId in cocoIds:
        path = os.path.join(activations_dir, f"subj_{subj:02d}", f"{cocoId:012d}.npz")
        if not os.path.exists(path):
            continue
        data = np.load(path)
        if layer not in data:
            continue
        features.append(data[layer].squeeze().flatten())
        valid_ids.append(cocoId)

    if len(features) < 2:
        print(f"Skipping {layer} due to insufficient data")
        continue

    # only keep centers for valid_ids
    centers = np.vstack([center_dict[cid] for cid in valid_ids])
    # normalize centers so that offsets lie roughly in [-0.5, +0.5]
    ctr_mean = centers.mean(axis=0)
    ctr_range = centers.ptp(axis=0)
    centers_norm = (centers - ctr_mean) / ctr_range

    # compute RDM + MDS
    X = np.vstack(features)
    rdm = squareform(pdist(X, metric="correlation"))
    mds = MDS(n_components=2, dissimilarity="precomputed", random_state=42)
    X_mds = mds.fit_transform(rdm)

    # now plot
    plt.figure(figsize=(6,6))
    # base MDS scatter
    plt.scatter(X_mds[:,0], X_mds[:,1], c='k', s=15, label='MDS points')

    # draw a little arrow for each point indicating bbox‐center offset
    plt.quiver(
        X_mds[:,0], X_mds[:,1],                   # arrow origins  
        centers_norm[:,0], centers_norm[:,1],     # arrow directions
        angles='xy', scale_units='xy', scale=5,   # tweak scale to taste
        width=0.002, color='tab:blue', alpha=0.7,
        label='bbox‐center offset'
    )

    # annotate with cocoId or however you like
    for i, cid in enumerate(valid_ids):
        continue
        plt.text(X_mds[i,0], X_mds[i,1], cid, fontsize=6, va='center')

    plt.title(f"MDS + bbox‐center for {layer}")
    plt.xlabel("MDS 1")
    plt.ylabel("MDS 2")
    plt.legend(loc='upper right', fontsize=6)
    plt.tight_layout()

plt.show()


In [ ]:
# build an area dict: cocoId → bbox area
area_dict = {}
for det in face_detections:
    cid = int(det["file_name"].lstrip("0")[:-4])
    if cid in valid_ids and len(det["detection"]) == 1:
        x1, y1, x2, y2 = det["detection"][0]
        area_dict[cid] = (x2 - x1) * (y2 - y1)

# collect areas in the same order as valid_ids
areas = np.array([area_dict[cid] for cid in valid_ids])

# Plot: MDS scatter, colored by bbox area, with a ‘hot’ colormap
plt.figure(figsize=(6,6))
sc = plt.scatter(
    X_mds[:,0], X_mds[:,1],
    c=areas,
    cmap='hot',
    s=30,               # fixed marker size; or use s=areas_norm*100 for bubble‐size
    # edgecolor='',
    alpha=0.8
)
cbar = plt.colorbar(sc)
cbar.set_label("BBox area (pixels²)")
plt.title(f"MDS embedding colored by face size for {layer}")
plt.xlabel("MDS 1")
plt.ylabel("MDS 2")
plt.tight_layout()
plt.show()


In [ ]:
import os
import json
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from sklearn.manifold import MDS
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Circle

# ——— set modern grid theme ———
sns.set_theme(style="whitegrid")

# ——— parameters ———
subj = 1
layer = "Sigmoid_217"
activations_dir = f"data/activations/detection"

# ——— load labels and coco IDs ———
faces_df = pd.read_excel(f"data/labels/subj_{subj:02d}/faces/faces_final.xlsx")
cocoIds = faces_df["cocoId"].astype(int).tolist()

# ——— load face detections ———
with open(f"data/labels/subj_{subj:02d}/face_detection_result.json", "r") as f:
    face_detections = json.load(f)

# ——— build dict: cocoId → bbox center ———
center_dict = {}
for det in face_detections:
    cid = int(det["file_name"][:-4].lstrip("0"))
    if cid in cocoIds and len(det["detection"]) == 1:
        x1, y1, x2, y2 = det["detection"][0]
        center_dict[cid] = np.array([ (x1 + x2) / 2, (y1 + y2) / 2 ])

# ——— load activation features ———
features = []
valid_ids = []
for cid in cocoIds:
    path = os.path.join(activations_dir, f"subj_{subj:02d}", f"{cid:012d}.npz")
    if not os.path.exists(path):
        continue
    data = np.load(path)
    if layer not in data:
        continue
    features.append(data[layer].squeeze().flatten())
    valid_ids.append(cid)

# ——— normalize centers for valid IDs ———
centers = np.vstack([center_dict[c] for c in valid_ids])
ctr_mean = centers.mean(axis=0)
ctr_range = centers.ptp(axis=0)
centers_norm = (centers - ctr_mean) / ctr_range

# ——— compute MDS embedding ———
X = np.vstack(features)
rdm = squareform(pdist(X, metric="correlation"))
mds = MDS(n_components=2, dissimilarity="precomputed", random_state=42)
X_mds = mds.fit_transform(rdm)

# ——— build dict: cocoId → bbox area ———
area_dict = {}
for det in face_detections:
    cid = int(det["file_name"][:-4].lstrip("0"))
    if cid in valid_ids and len(det["detection"]) == 1:
        x1, y1, x2, y2 = det["detection"][0]
        area_dict[cid] = (x2 - x1) * (y2 - y1)
areas = np.array([area_dict[c] for c in valid_ids])

# ——— continuous pastel colormap for right plot ———
pastel_cmap = sns.color_palette("pastel", as_cmap=True)

# ——— create figure with two subplots ———
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax in axes:
    # draw unit circle
    circ = Circle((0, 0), radius=1.0,
                  fill=False, lw=1,
                  color='gray', alpha=0.7)
    ax.add_patch(circ)
    # fix limits and enforce equal aspect
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_aspect('equal', 'box')

# ——— left subplot: black points + blue arrows ———
ax = axes[0]
ax.scatter(
    X_mds[:, 0], X_mds[:, 1],
    c='k', s=25, edgecolors='w', label='MDS points'
)
ax.quiver(
    X_mds[:, 0], X_mds[:, 1],
    centers_norm[:, 0], centers_norm[:, 1],
    angles='xy', scale_units='xy', scale=5,
    width=0.003, color='tab:blue', alpha=0.7,
    label='BBox-center offset'
)
ax.set_title(f"MDS + bbox-center for {layer}", fontsize=12)
ax.set_xlabel("MDS 1")
ax.set_ylabel("MDS 2")
ax.legend(fontsize=8)

# ——— right subplot: continuous pastel colormap by area ———
ax = axes[1]
sc = ax.scatter(
    X_mds[:, 0], X_mds[:, 1],
    c=areas, cmap="hot",
    s=40, alpha=0.8, edgecolors='w'
)
cbar = fig.colorbar(sc, ax=ax, shrink=0.85, pad=0.02)
cbar.set_label("BBox area (pixels²)")
ax.set_title(f"MDS colored by face size for {layer}", fontsize=12)
ax.set_xlabel("MDS 1")
ax.set_ylabel("MDS 2")

plt.tight_layout()
plt.show()


In [ ]:
"""

Relu_2
  Relu_5
  Relu_8
  Relu_12
  Relu_16
  Relu_19
  Relu_23
  Relu_26
  Relu_30
  Relu_33
  Relu_40
  Relu_43
  Relu_47
  Relu_50
  Relu_54
  Relu_57
  Relu_61
  Relu_64
  Relu_71
  Relu_74
  Relu_78
  Relu_81
  Relu_88
  Relu_91
  Relu_95
  Relu_98
  Relu_102
  Relu_157
  Relu_160
  Relu_163
  Sigmoid_171
  Relu_180
  Relu_183
  Relu_186
  Sigmoid_194
  Relu_203
  Relu_206
  Relu_209
  Sigmoid_217

"""




import os
import glob
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from sklearn.manifold import MDS
from rsa.permutation_analysis import run_mantel_test
# assume run_mantel_test is already defined/imported
# assume metadata is a DataFrame or structure needed by run_mantel_test

subj = 1
activations_dir = "data/activations"
faces_df = pd.read_excel(f"data/labels/subj_{subj:02d}/faces/faces_final.xlsx")
cocoIds = faces_df["cocoId"].tolist()
metadata = [f"{x:012d}" for x in cocoIds]


# allowed_modules = ["detection", "genderage"]
# allowed_modules = ["detection", "genderage"]
allowed_modules = ["detection"]

results = []
B = 2000

out_path = f"neural_net_results_subj_{subj:02d}.xlsx"


if not os.path.exists(out_path):

    for module in allowed_modules:
        print(f"=== Module: {module} ===")
        subj_dir = os.path.join(activations_dir, module, f"subj_{subj:02d}")
        npz_files = glob.glob(os.path.join(subj_dir, "*.npz"))
        if not npz_files:
            print(f"No .npz files found in {subj_dir}, skipping.")
            continue
        
        # 1) Gather full set of layers across all files
        all_layers = set()
        for path in npz_files:
            data = np.load(path)
            all_layers.update(data.files)

        if module == "detection":
            # all_layers = ['Relu_64', 'Relu_98', 'Relu_157','Relu_180', "Sigmoid_217", "Sigmoid_194", "Relu_203"]
            all_layers = ['Relu_30']
            
        elif module == "genderage":
            all_layers = ['conv_8_relu',
    'conv_9_dw_relu',
    'conv_9_relu',
    'conv_10_dw_relu',
    'conv_10_relu',
    'conv_11_dw_relu',
    'conv_11_relu',
    'conv_12_dw_relu',
    'conv_12_relu',
    'conv_13_dw_t0_relu',
    'conv_13_t0_relu',
    'conv_14_dw_t0_relu',
    'conv_14_t0_relu',
    'conv_13_dw_t1_relu',
    'conv_13_t1_relu',
    'conv_14_dw_t1_relu']
        
        print(f"Found {len(all_layers)} layers: {sorted(all_layers)}")

        # 2) Process each layer
        for layer in tqdm(sorted(all_layers)):
            if module == "detection":
                feauture_selections = ["centers", "sizes"]
            elif module == "genderage":
                feauture_selections = ["feature_array_age", "genders"]

            for feauture_selection in feauture_selections:
                features = []
                valid_ids = []
                for cocoId in cocoIds:
                    path = os.path.join(subj_dir, f"{cocoId:012d}.npz")
                    if not os.path.exists(path):
                        continue
                    data = np.load(path)
                    if layer not in data:
                        continue
                    feat = data[layer].squeeze().flatten()
                    features.append(feat)
                    valid_ids.append(cocoId)

                if len(features) < 2:
                    print(f"  [layer={layer}] only {len(features)} valid items → skipping")
                    continue

                # 3) Compute RDM and MDS
                X = np.vstack(features)
                print(f"Dimensionality: {X.shape}")
                rdm = squareform(pdist(X, metric="correlation"))
                mds = MDS(n_components=2, dissimilarity="precomputed", random_state=42)
                X_mds = mds.fit_transform(rdm)

                # 4) Run Mantel test
                mantel_out = run_mantel_test(X_mds, subj, metadata, B=B, feature_selection=feauture_selection)
                # example mantel_out: {'subject':1, 'r_obs':..., 'effect_size':..., 'p_value':...}

                # 5) Record results
                results.append({
                    "subject":      mantel_out.get("subject", subj),
                    "module":       module,
                    "layer":        layer,
                    "n_images":     len(valid_ids),
                    "r_obs":        round(mantel_out["r_obs"], 3),
                    "p_value":      round(mantel_out["p_value"], 3),
                    "B":            B,
                    "feature_selection": feauture_selection
                })

        print()  # blank line between modules

    # 6) Build DataFrame and save
    df_results = pd.DataFrame(results)
    print("Final results:")
    print(df_results)

    # save to disk
    df_results.to_excel(out_path, index=False)
    print(f"Saved Mantel test results to {out_path}")


In [ ]:
from utils.utils import retrieve_roi_mask_extended
from utils.config import load_config

config = load_config("config.yaml")
retrieve_roi_mask_extended(config, 1, "subj_01", True)